In [6]:
import os

--------

# Deploying a Simple Model with FastAPI and curl 

## 1. Goal of this notebook
This notebook demonstrates the basic idea of deploying a machine learning model as an API and interacting with it using curl. The model itself is deliberately simple. The focus is on understanding:

- What it means to serve a model
- How HTTP requests work
- How to use curl to interact with an API
- How data is sent to a model and predictions are returned

In some production systems ML models are deployed as web APIs that other systems can query. There are other options for model deployment but this framework is popular because it is good for a wide variety of services to interact with the model.

In this notebook we will:
1. Run a small FastAPI application that hosts a model
2. Send requests to it using curl
3. Examine how the request and response work

--------

## 2. What is an API?

An **API (Application Programming Interface)** is a way for different pieces of software to communicate with each other. Instead of one program directly accessing another program’s internal code, the interaction happens through a clearly defined interface.

A useful analogy is a restaurant:
- The customer (the client) chooses something from the menu
- The waiter (API) takes the order to the kitchen
- The kitchen (backend service/model) prepares the food
- The waiter (API) returns the food to the customer

Just as the customer does not need to know how the kitchen functions, the client does not need to know how the backend service works. It simply needs to know how to place an order correctly. If it achieves that then it's request can be sent to the service. 

#### APIs for machine learning

In the context of machine learning, the order represents data that are sent to the model for a prediction to be made which is then transferred back to the client as a response. Typically the APIs are web APIs which means that they communicate over the internet using HTTP requests. These include:

- a URL (the address of the API)
- an HTTP method (e.g. GET or POST; this is discussed later)
- optional data, often formatted as JSON

The response then comes back as a JSON.

#### What is FastAPI?

FastAPI is a Python framework for building APIs. It allows us to create endpoints quickly that other systems can interact with, taking care of may tasks automatically:

- receiving HTTP requests
- parsing JSON data
- validating input data
- running Python functions
- returning responses in JSON format

For example, the following FastAPI code defines an API endpoint that accepts prediction requests:

```Python
@app.post("/predict")
def predict(data: IrisFeatures):
```

This tells FastAPI:
> When a POST request is sent to /predict, run the predict function using the provided input data.

FastAPI then handles all the underlying work needed to connect the HTTP request to the Python function.

It also automatically generates interactive documentation which can be accessed at `http://127.0.0.1:8000/docs`

#### Running the API server

We now run the API that hosts the model. We will reference the "DeploymentTest.py" script which trains a model on the Iris dataset, then loads that model, waits for incoming requests and runs predictions when receiving data.

Once the server is running, it listens on a specific port on your machine. In this example the address is `http://127.0.0.1:8000` meaning:

- `127.0.0.1` → your own computer (localhost)
- `8000` → the port number the server is listening on

The way to run the API server is to run the below code in the terminal. You should do this in a separate terminal to the one in which you want to call the API as we need to let it keep running and the server will continue running until it is stopped (achieved with CTRL+C). In order to run the commands in the below codeblocks, make sure to run the API server with the below code in a terminal window.

```Linux
uvicorn DeploymentTest:app --reload
```

--------

## 3. What is curl?

`curl` is a command line tool that allows you to send HTTP requests to a server. This is useful for interacting with our web API described above.

Normally a browser will send requests like: `GET https://example.com`. However, many APIs expect POST requests with JSON data, which browsers cannot easily send directly from the address bar. `curl` allows us to manually construct these requests.

#### Basic structure of a curl command

A typical curl request looks like this:

`curl -X POST URL -H "HEADER" -d "DATA"`

where each section has its own role.

| Component | Meaning |
| ---- | ---- |
| `curl` | Runs the curl program |
| `-X POST` | The HTTP method |
| `URL` | The address of the API |
| `-H` | HTTP headers |
| `-d` | Data sent in the request |

Because APIs usually expecte JSON data, we often include the following in the header to tell the server that the request body contains JSON: `Content-Type: application/json`

------------

## 4. Interacting with our API

#### Testing that the server is running

Once the server is running, we can send a simple request. If the API includes a root endpoint (/), we can test it with the below command which sends a GET request to the server. If the server is running correctly, it should return a response. You should also see a new line in the terminal window where you started the server running, which indicates that the API has received a GET call.

If you stop the server running and rerun the below code then you will get a message saying that a connection to the server was not possible. This is a good demonstration of GET being a good check that the server is running - in the `DeploymentTest.py` file we have called the function `health_check` for this reason.

In [19]:
os.system("curl http://127.0.0.1:8000")

{"status":"ok"}

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    15  100    15    0     0   8269      0 --:--:-- --:--:-- --:--:-- 15000


0

In [20]:
# We add -s here to suppress the progress bar output from curl
# There is no need to do this in the terminal
os.system("curl -s http://127.0.0.1:8000")

{"status":"ok"}

0

#### Sending data to the model

Our model exposes an endpoint called `/predict`. This endpoint expects:
- A POST request
- JSON input data containing the model features

The JSON structure must match the schema expected by the API where we have defined this with the `IrisFeatures` class which inehrits from the Pydantic `BaseModel` class. When receiving this request the server will:
1. Receive this JSON data
2. Convert it into Python objects
3. Run the model prediction
4. Return the predicted class

To send data to the model, we use a POST request. The curl command contains:
- `-X POST` → specify the HTTP method
- `-H "Content-Type: application/json"` → tell the server the data format
- `-d` → the JSON data

In [26]:
os.system(
    """
    curl -s -X POST "http://127.0.0.1:8000/predict" \
     -H "Content-Type: application/json" \
     -d '{
       "sepal_length": 5.1,
       "sepal_width": 3.5,
       "petal_length": 1.4,
       "petal_width": 0.2
     }'
     """
)

{"class":"setosa"}

0

#### What happens when curl sends a request?

When you run the curl command, the following happens:
1. `curl` sends an HTTP request to the API
2. The FastAPI server receives the request
3. FastAPI validates the JSON input
4. The model runs a prediction
5. The API sends back a JSON response

It is worth playing round with the above request and seeing if you can break it. For example, below we pass a non-float data type and we get an error because it does not match the type expected by our API. Have a look at the new line in the terminal to see that we get a useful message there too.

In [30]:
os.system(
    """
    curl -s -X POST "http://127.0.0.1:8000/predict" \
     -H "Content-Type: application/json" \
     -d '{
       "sepal_length": "test",
       "sepal_width": 3.5,
       "petal_length": 1.4,
       "petal_width": 0.2
     }'
     """
)

{"detail":[{"type":"float_parsing","loc":["body","sepal_length"],"msg":"Input should be a valid number, unable to parse string as a number","input":"test"}]}

0

--------

## 5. Summary

In this notebook we learned:
- What it means to deploy a model as an API
- How FastAPI exposes prediction endpoints
- How curl can send HTTP requests to an API
- How data is sent as JSON
- How the server returns predictions

The key idea is that deployment allows other systems to request predictions from a model.